# Qwen2.5-7B-Instruct — MLX variants summary

Loads the FP16 / 4-bit / 2-bit result JSONs from `data/`, reconstructs the `name` + `variant` fields from each run's label, and renders the KPI table + writes `summary.csv` via `bench_utils.print_results_table` and `bench_utils.dump_results_table`.

In [1]:
import json
import logging
import re
import sys
from pathlib import Path

# Make the repo root importable so we can pull the summary helpers.
REPO_ROOT = Path.cwd().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from bench_utils import (
    dump_results_table,
    print_latex_legend_table,
    print_latex_table,
    print_results_table,
)

# print_results_table logs via the `bench` logger — wire it to stdout so the
# table shows up in the notebook cell output.
bench_log = logging.getLogger("bench")
bench_log.setLevel(logging.INFO)
if not bench_log.handlers:
    h = logging.StreamHandler(sys.stdout)
    h.setFormatter(logging.Formatter("%(message)s"))
    bench_log.addHandler(h)

DATA_DIR = Path.cwd() / "data"
print("data dir:", DATA_DIR)

/Users/bruski/.pyenv/versions/3.12.7/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data dir: /Users/bruski/workspace/LLM-Inference-Benchmarking-2/analysis/comparisons/data


In [2]:
# Label format from benchmark_mlx.py is `f"{variant} ({name})"`.
LABEL_RE = re.compile(r"^(?P<variant>.+?)\s*\((?P<name>[^)]+)\)\s*$")

# Preferred display order — FP16 first, then ascending bitwidth.
VARIANT_ORDER = {"FP16": 0, "MLX 4-bit": 1, "MLX 2-bit": 2}

def load_run(path: Path) -> dict:
    payload = json.loads(path.read_text())
    m = LABEL_RE.match(payload["label"])
    if not m:
        raise ValueError(f"unparseable label in {path.name}: {payload['label']!r}")
    row = dict(payload["results"])
    row["name"] = m.group("name")
    row["variant"] = m.group("variant")
    return row

rows = [load_run(p) for p in sorted(DATA_DIR.glob("*.json"))]
rows.sort(key=lambda r: (r["name"], VARIANT_ORDER.get(r["variant"], 99)))

for r in rows:
    print(f"  {r['variant']:<10}  {r['name']}")

  MLX 4-bit   Gemma-2-9B-it
  MLX 2-bit   Gemma-2-9B-it
  FP16        Llama-3.1-8B-Instruct
  MLX 4-bit   Llama-3.1-8B-Instruct
  MLX 2-bit   Llama-3.1-8B-Instruct
  FP16        Qwen2.5-7B-Instruct
  MLX 4-bit   Qwen2.5-7B-Instruct
  MLX 2-bit   Qwen2.5-7B-Instruct


In [3]:
print_results_table(rows)


  SUMMARY (all models are instruct-tuned; GSM8K n=10)
Model                 Quant.        Acc.       PPL            Speed                TTFT            TPOT           Latency       Weight       Runtime       Peak
                                                            (Tok/s)                (ms)            (ms)               (s)         (GB)          (GB)       (GB)
---------------------------------------------------------------------------------------------------------------------------------------------------------------
Gemma-2-9B            MLX 4-bit      90%       6.8       37.0 ± 1.2        457.5 ± 45.1      23.9 ± 0.5         3.9 ± 0.9          4.8           0.3        5.1
Gemma-2-9B            MLX 2-bit       0%      31.2       53.2 ± 0.4        480.8 ± 58.4      17.9 ± 0.0         9.6 ± 0.1          2.7           0.3        3.0
Llama-3.1-8B          FP16           80%       5.7       15.4 ± 0.4        476.7 ± 13.5      62.6 ± 1.1        13.9 ± 7.0         15.0           

In [ ]:
# AAAI-friendly LaTeX output — paste both tables into the paper.
# Requires \usepackage{booktabs} in the LaTeX preamble.
# ID convention: <model-initial>-<framework-initial>-<bits>, e.g. L-M-16
# = Llama on MLX at FP16. Legend table defines every ID used below.
print_latex_legend_table(
    rows,
    caption="Model configurations evaluated. All variants are instruct-tuned.",
    label="tab:mlx-legend",
)
print()
print_latex_table(
    rows,
    caption="MLX quantization benchmark results on 10 GSM8K samples. "
            "See Table~\\ref{tab:mlx-legend} for the ID key.",
    label="tab:mlx-results",
)

In [5]:
# Write the union-of-keys CSV next to this notebook.
import os
output_dir = os.path.join(str(Path.cwd()), 'outputs')
os.makedirs(output_dir, exist_ok=True)
out_path = dump_results_table(output_dir, rows, filename="qwen_summary.csv")
print("wrote:", out_path)

wrote summary csv: /Users/bruski/workspace/LLM-Inference-Benchmarking-2/analysis/comparisons/outputs/qwen_summary.csv (8 rows)
wrote: /Users/bruski/workspace/LLM-Inference-Benchmarking-2/analysis/comparisons/outputs/qwen_summary.csv


In [6]:
# Optional: preview as a pandas DataFrame for interactive filtering/sorting.
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    front = ["name", "variant", "gsm8k_accuracy", "perplexity", "throughput_tok_s", "ttft_mean_ms", "tpot_mean_ms", "weight_mem_mb", "peak_mem_mb"]
    df = df[front + [c for c in df.columns if c not in front]]
    display(df)
except ImportError:
    print("pandas not installed — skipping DataFrame preview")

,name,variant,gsm8k_accuracy,perplexity,throughput_tok_s,ttft_mean_ms,tpot_mean_ms,weight_mem_mb,peak_mem_mb,throughput_mean_tok_s,...,gsm8k_correct,gsm8k_total,runtime_mem_mb,framework,engine,engine_version,quant_method,quant_bits,quant_format,kernel
0,Gemma-2-9B-it,MLX 4-bit,0.9,6.806845,37.197072,457.513233,23.892433,4958.467781,5221.217199,36.981324,...,9,10,262.749418,MLX,mlx-lm,0.31.2,affine,4,affine-gs64,mlx
1,Gemma-2-9B-it,MLX 2-bit,0.0,31.161373,53.162491,480.754550,17.906254,2755.217800,3103.371946,53.164931,...,0,10,348.154146,MLX,mlx-lm,0.31.2,affine,2,affine-gs64,mlx
2,Llama-3.1-8B-Instruct,FP16,0.8,5.714682,15.531226,476.727742,62.629761,15316.509285,15473.688042,15.418854,...,8,10,157.178757,MLX,mlx-lm,0.31.2,None,16,None,mlx
3,Llama-3.1-8B-Instruct,MLX 4-bit,0.8,6.227768,49.017484,399.268979,18.442388,4308.134285,4559.594440,48.812162,...,8,10,251.460155,MLX,mlx-lm,0.31.2,affine,4,affine-gs64,mlx
4,Llama-3.1-8B-Instruct,MLX 2-bit,0.0,1001.870308,71.125800,400.581933,13.303192,2393.634285,2743.078873,71.147644,...,0,10,349.444588,MLX,mlx-lm,0.31.2,affine,2,affine-gs64,mlx
5,Qwen2.5-7B-Instruct,FP16,0.8,5.082452,16.700983,441.356925,58.164842,14525.635750,14651.995644,16.677604,...,8,10,126.359894,MLX,mlx-lm,0.31.2,None,16,None,mlx
6,Qwen2.5-7B-Instruct,MLX 4-bit,0.8,5.609574,53.147520,367.434392,17.381186,4088.885750,4331.511406,52.941635,...,8,10,242.625656,MLX,mlx-lm,0.31.2,affine,4,affine-gs64,mlx
7,Qwen2.5-7B-Instruct,MLX 2-bit,0.0,13421.872589,75.549643,375.953342,12.526512,2273.260750,2584.573963,75.582817,...,0,10,311.313213,MLX,mlx-lm,0.31.2,affine,2,affine-gs64,mlx
